Montgomery County Maryland Wine Sales Dashboard V2

Module imports

In [2]:
import pandas as pd
import numpy as np
import requests
import PyPDF2
import io
import re
import sqlite3
import time
import datetime
import plotly.graph_objects as go
import plotly.express as px
import warnings
import sys
import pickle
import os
import streamlit as st
from plotly.subplots import make_subplots
from difflib import SequenceMatcher
from datetime import datetime
from openpyxl import Workbook
from io import StringIO
import importlib

# Add utils path FIRST
sys.path.append('./utils')

# Import custom utilities
import fuzzy_supplier_matching as fuzzy
import enhanced_data_cleaning_utils as deu
import wine_classification_utils as wcu
import wine_review_matching_utils as wrmu
import inspect

# Reload fuzzy module after fixing indentation
importlib.reload(fuzzy)

warnings.filterwarnings('ignore')

NameError: name 'df_classified' is not defined

File imports

In [ ]:
# GitHub raw URL
base_url = "https://raw.githubusercontent.com/ac604605/Montgomery_County_Dashboard/main/"

print("Loading datasets from GitHub repository...")
print("=" * 60)

try:
    # Load standard datasets
    Distributors_Virginia_Three_Main = pd.read_csv(base_url + "data/Distributors_Virginia_Three_Main.csv")
    wine_producers = pd.read_csv(base_url + "data/wine_producers.csv")
    Warehouse_and_Retail_Sales = pd.read_csv(base_url + "data/Warehouse_and_Retail_Sales.csv")
    Wine_Review_Data = pd.read_csv(base_url + "data/winemag-data-130k-v2.csv/winemag-data-130k-v2.csv")
    
    # Load suppliers with data quality fix
    print("Loading and fixing supplier data structure...")
    correct_columns = ['License_ID', 'Trade Name', 'Address', 'City', 'State', 'Zip_Code', 'Report_Type']
    Suppliers_Fixed = pd.read_csv(
        base_url + "data/Suppliers_Importers_Retailers.csv",
        header=0,
        names=correct_columns,
        usecols=range(7),
        dtype={'License_ID': str, 'Zip_Code': str}
    )
    
    # Professional summaries
    datasets = [
        (Distributors_Virginia_Three_Main, "Virginia Distributors"),
        (wine_producers, "Wine Producers"),
        (Warehouse_and_Retail_Sales, "Sales Transactions"),
        (Wine_Review_Data, "Wine Reviews"),
        (Suppliers_Fixed, "Supplier Directory (Fixed)")  # Note the "Fixed" indicator
    ]
    
    for df, name in datasets:
        print(f"{name:<25} │ {df.shape[0]:>8,} rows × {df.shape[1]:>2} cols │ {df.memory_usage(deep=True).sum() / 1024**2:>6.1f} MB")
    
    # Quick validation for suppliers
    report_types = Suppliers_Fixed['Report_Type'].nunique()
    print(f"Supplier validation: {report_types} unique report types identified")
    
    print("=" * 60)
    print(f"Successfully loaded {len(datasets)} datasets with data quality fixes applied")
    
except Exception as e:
    print(f"Error loading data: {e}")

Now with everything loaded, I will begin data cleaning and refinement to suit the needs of this specific dashboard. 

After investigating the sales data, there are a few cleaning steps that need to take place. First will be removing all values that are not wine and beer items carried by distributors. Second will be ensuring item codes are numeric for easier processesing. Finally, some idividual values will be changed and anything that isn't wine or beer will be removed. I will also be removing the keg versions of wines and beer, as those would introduce greater scope that I do not wish to manage for a simple portfolio. 

In [ ]:
# Create working copy for processing
print("Creating working copy of sales data...")
df_working = Warehouse_and_Retail_Sales.copy()
print(f"Working dataset: {df_working.shape[0]:,} rows × {df_working.shape[1]} columns")

# Run your enhanced data cleaning utilities
print("\nStarting data cleaning pipeline...")
df_clean, cleaning_report = deu.run_complete_item_code_standardization(
    df_working, 
    item_types_to_keep=['WINE', 'BEER']
)

# Show cleaning results
print(f"\nCleaning Results:")
print(f"   Original: {cleaning_report['original_shape']}")
print(f"   Cleaned:  {cleaning_report['final_shape']}")
print(f"   Retention: {cleaning_report['summary']['data_retention_pct']:.1f}%")

I will also perform some joins and various cleaning of supporting data tables meant to make brand ownership rights more clear. 

In [ ]:
# Run supplier enrichment with fuzzy matching
print("Starting supplier enrichment pipeline...")
df_enriched = fuzzy.run_supplier_enrichment(
    df_clean, 
    Suppliers_Fixed, 
    test_mode=False
)

# Show enrichment results
print(f"\nSupplier Enrichment Results:")
matched_suppliers = df_enriched[
    (df_enriched['SUPPLIER_MATCH_SCORE'] >= 0.8) & 
    (df_enriched['SUPPLIER_REPORT_TYPE'] == 'Wholesale Wine Distributors')
]
match_rate = len(matched_suppliers) / len(df_enriched) * 100
print(f"   Rows matched to distributors: {len(matched_suppliers):,}/{len(df_enriched):,} ({match_rate:.1f}%)")
print(f"   Unique distributors identified: {matched_suppliers['MATCHED_SUPPLIER_NAME'].nunique()}")

# Quick preview of top distributors
print(f"\nTop 5 Wholesale Wine Distributors by volume:")
print(matched_suppliers['SUPPLIER'].value_counts().head())

After reviewing the wine review data, it appears this will serve as an adequate database to gather missing country data for our sales table. For the sake of simplicity, I will combine all columns from this table to matching wines in our sales table. Columns we do not need can be filtered out later. 

full_results, sales_map, review_map = wrmu.run_wine_review_matching(
    df_clean_Warehouse_and_Retail_Sales, Wine_Review_Data, threshold=0.6, test_mode=False
)

In [ ]:
# Save as pickle (recommended for data analysis)
#full_results.to_pickle('wine_sales_with_reviews_FINAL.pkl')

# To load later:
df = pd.read_pickle('wine_sales_with_reviews_FINAL.pkl')

In [ ]:
#uncomment below lines to change dataframe.
#df=Warehouse_and_Retail_Sales
#df=Distributors_Virginia_Three_Main
#df=Wine_Review_Data
#df=df_clean_Warehouse_and_Retail_Sales_enhanced
#df=Suppliers_Fixed
#df=df_loaded
#df=df_final
df=df

#uncomment below lines to change report
#df.head(25)
df.info()
#df.describe()
#df[df['final_variety'] == ''].head()
#pd.set_option('display.max_rows', None)
#df['ITEM DESCRIPTION'].value_counts()
#deu.investigate_missing_values_manually(df)

In [ ]:
# After wine review matching, add classification
print("Starting comprehensive wine classification...")
df_classified, variety_counts, country_counts = wcu.run_enhanced_wine_classification(df)

# Add wine color classification
print("Adding wine color classification...")
df_classified['wine_color'] = df_classified['final_variety'].apply(wcu.classify_wine_color)
print("Wine color distribution:")
print(df_classified['wine_color'].value_counts())

# Quick summary
wcu.quick_enhanced_summary(df_classified)

# Final enriched dataset
df_complete = df_classified
print(f"Complete pipeline finished! Dataset: {df_complete.shape}")

In [ ]:
# Save as pickle (recommended for data analysis)
df_classified.to_pickle('wine_data_fully_classified.pkl')

# To load later:
#df = pd.read_pickle('wine_data_fully_classified.pkl')